In [1]:
import os
import torch
import tifffile
import numpy as np
import pandas as pd
import torch.nn.functional as F
from pathlib import Path
from napari_cellseg3d.code_models.models.wnet.model import WNet

# ================= CONFIGURATION =================
CONFIG = {
    # 1. PATHS
    "MODEL_PATH": r"/work/scratch/rawat/models/wnet3d/wnet_lr2e-05_patch64_sigS4.0_sigI1.0_rad2/model_final.pth",
    "INPUT_FOLDER": r"/home/students/rawat/czi_output/processed_images/Processed_images_tif",
    "OUTPUT_FOLDER": r"/work/scratch/rawat/results/Wnet_segmentations",
    
    # 2. MICROSCOPE METADATA (Z, Y, X)
    "VOXEL_SIZE": (0.23, 0.1418360, 0.1418360), 
    
    # 3. SETTINGS
    "DEVICE": "cpu"
}
# =================================================

def calculate_volume(mask_array, voxel_size):
    """Calculates volume of the segmentation mask in cubic microns."""
    pixel_count = np.sum(mask_array > 0)
    voxel_vol = voxel_size[0] * voxel_size[1] * voxel_size[2]
    return pixel_count * voxel_vol

def pad_to_divisor(img_tensor, divisor=16):
    """
    Pads spatial dims to be divisible by 16.
    Required for WNet/U-Net to function without crashing on odd sizes.
    """
    d, h, w = img_tensor.shape[2], img_tensor.shape[3], img_tensor.shape[4]
    
    target_d = int(np.ceil(d / divisor) * divisor)
    target_h = int(np.ceil(h / divisor) * divisor)
    target_w = int(np.ceil(w / divisor) * divisor)
    
    pad_d = target_d - d
    pad_h = target_h - h
    pad_w = target_w - w
    
    # Pad order: (Left, Right, Top, Bottom, Front, Back)
    padded_tensor = F.pad(img_tensor, (0, pad_w, 0, pad_h, 0, pad_d))
    
    return padded_tensor, (d, h, w)

def load_image_robust(path):
    """Load, Normalize (High Contrast), and Pad"""
    img = tifffile.imread(path).astype(np.float32)
    
    # 1. Standardize dimensions (C, Z, Y, X)
    if img.ndim == 3: img = img[np.newaxis, ...]
    elif img.ndim == 4:
        if img.shape[1] <= 4: img = np.transpose(img, (1, 0, 2, 3))
        elif img.shape[3] <= 4: img = np.transpose(img, (3, 0, 1, 2))
    
    # 2. Keep 2 channels
    if img.shape[0] >= 2: img = img[0:2, ...]
    
    # 3. Normalize (YOUR CUSTOM PERCENTILES)
    for c in range(img.shape[0]):
        # Using 0.0001 and 99.999 as requested
        low, high = np.percentile(img[c], (0.0001, 99.999))
        if high > low:
            img[c] = np.clip((img[c]-low)/(high-low), 0, 1)
        else:
            img[c] = np.zeros_like(img[c])
            
    # 4. Add Batch Dim
    tensor = torch.tensor(img).unsqueeze(0)
    
    # 5. PAD (The Fix for the crash)
    padded_tensor, original_shape = pad_to_divisor(tensor, divisor=16)
    
    return padded_tensor, original_shape

def run_inference():
    os.makedirs(CONFIG["OUTPUT_FOLDER"], exist_ok=True)
    
    print(f"Loading WNet model from {CONFIG['MODEL_PATH']}...")
    print(f"Using Device: {CONFIG['DEVICE']}")
    
    # Initialize Model
    model = WNet(in_channels=2, out_channels=2, num_classes=2).to(CONFIG["DEVICE"])
    
    # Load Weights
    checkpoint = torch.load(CONFIG["MODEL_PATH"], map_location=CONFIG["DEVICE"])
    model.load_state_dict(checkpoint)
    model.eval()
    
    image_paths = sorted(list(Path(CONFIG["INPUT_FOLDER"]).glob("*.tif")))
    results_data = []
    
    print(f"Processing {len(image_paths)} images...")
    
    with torch.no_grad():
        for p in image_paths:
            # 1. Load Padded
            img_tensor, original_shape = load_image_robust(p)
            img_tensor = img_tensor.to(CONFIG["DEVICE"])
            
            # 2. Forward Pass
            seg_out, _ = model(img_tensor)
            
            # 3. Get Probability Map
            pred_prob = seg_out[0, 0, ...].cpu().numpy()
            
            # 4. CROP BACK (Remove Padding)
            # We cut off the extra pixels we added so the output size matches input size
            oz, oy, ox = original_shape
            pred_prob = pred_prob[:oz, :oy, :ox]
            
            # 5. Threshold & Volume
            mask = (pred_prob > 0.5).astype(np.uint8)
            vol = calculate_volume(mask, CONFIG["VOXEL_SIZE"])
            
            results_data.append({"filename": p.name, "volume_um3": vol})
            
            # Save Mask
            save_path = os.path.join(CONFIG["OUTPUT_FOLDER"], f"mask_{p.name}")
            tifffile.imwrite(save_path, mask)
            print(f"  Saved: {p.name} | Volume: {vol:.2f} µm³")

    # Save CSV
    df = pd.DataFrame(results_data)
    csv_path = os.path.join(CONFIG["OUTPUT_FOLDER"], "volume_stats.csv")
    df.to_csv(csv_path, index=False)
    print(f"\nDone! Volume statistics saved to: {csv_path}")

if __name__ == "__main__":
    run_inference()

Loading WNet model from /work/scratch/rawat/models/wnet3d/wnet_lr2e-05_patch64_sigS4.0_sigI1.0_rad2/model_final.pth...
Using Device: cpu
Processing 3 images...
  Saved: PN_KC_488_568_Brain1_CalyxR_LSMPlus_16bit-1.tif | Volume: 36115.13 µm³


: 